# 発展課題1（演習1：スレッドとパイプライン）

3問あります。**全部やる必要はありません。** 関係の深いものだけ選んでください。

- [問1-1 ボトルネックと上限 FPS](#scrollTo=adv01_threads_01)（紙と鉛筆）
- [問1-2 `join()` を消すとどうなるか](#scrollTo=adv01_threads_03)（実験）
- [問1-3 コア数より多くスレッドを立てたら](#scrollTo=adv01_threads_07)（実験）

## 問1-1. ボトルネックと上限 FPS

各段の所要時間が **Read=13ms、Infer=67ms、Show=33ms** だとします。

- (a) **3フレーム分**を1スレッドで順番に処理すると何 ms か。3スレッドのパイプラインでは何 ms か
- (b) **どの段がボトルネックか。** 十分に長く流し続けたときの上限は何 FPS か
- (c) 真ん中だけを大幅に速くして **Infer=2ms** にしたとき、**ボトルネックはどこに移るか。**
  真ん中を33倍速くして、上限の FPS は何倍になるか

**紙の上でタイムラインを描いてから**、下の解答を見てください。

### 問1-1 の解答

**(a)** 1フレーム 13+67+33 = 113ms。

- 1スレッド … 113 × 3 = **339ms**
- 3スレッド … 113 + 67 × 2 = **247ms**（1フレーム目が出るまで113ms、以後 最遅段の67msごと）

**(b)** ボトルネックは **Infer（67ms）**。上限は `1000 / 67 =` **約 14.9 FPS**。

**(c)** Infer=2ms にすると各段は 13 / 2 / 33。**ボトルネックは Show（33ms）に移ります。**
上限は `1000 / 33 =` 約 30.3 FPS。

**真ん中を33倍速くしたのに、上限 FPS は 14.9 → 30.3 で約2倍にしかなりません。**

> **ボトルネック以外をどれだけ速くしても、全体はほとんど変わらない。**
> そして**ボトルネックを速くすると、ボトルネックは別の段へ移る。**
> だから改造は「1回に1つだけ変えて、また測る」の繰り返しになります。

**おまけ**：真ん中の段を2人に増やしたら？ (c) の状態では Infer は 2ms しかないので、
**まったく意味がありません**（演習3-2 の③「効いていない上限を上げても、何も起きない」）。
そして一般に、2人にしても倍にならない場合があります ―― たとえば**画面や送信先が1つしかない段**は、
2人にしても順番待ちになるだけです。

## 問1-2. `join()` を消すとどうなるか

`ex01a.cpp` から `t1.join(); t2.join();` を消すと何が起きるでしょうか。
**予測してから**実行してください。

In [ ]:
%%writefile adv01a.cpp
#include <iostream>
#include <thread>
#include <chrono>
using namespace std::chrono;

void job(const char* name) {
    std::this_thread::sleep_for(milliseconds(300));
    std::cout << name << " が終わった\n";
}

int main() {
    std::cout << "スレッドを2本立てる\n" << std::flush;
    std::thread t1(job, "仕事A");
    std::thread t2(job, "仕事B");
    // t1.join();  t2.join();      ← わざと消してある
    std::cout << "main が先に終わろうとしている\n" << std::flush;
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread adv01a.cpp -o adv01a
!./adv01a; echo "終了コード=$?"

### 問1-2 の解答 ―― 異常終了する

```
スレッドを2本立てる
main が先に終わろうとしている
terminate called without an active exception
Aborted
```

`std::thread` のオブジェクトが**まだ動いているスレッドを持ったまま**寿命を迎えると、
C++ は `std::terminate()` を呼びます。つまり**異常終了します。**

「仕事Aが終わった」も表示されません。`main` は 300ms 待たずに終わろうとするからです。

> **起動したスレッドは、必ずどこかで `join()` する。**
> `join()` が返らない／呼び忘れる、はどちらもプログラムを壊します。

## 問1-3. コア数より多くスレッドを立てたら

`hardware_concurrency()` が **2** のマシンで、**計算しかしない仕事**（1本あたり300ms分）を
持つスレッドを **3本** 立てます。

- (a) 3本目のスレッドは、コアが空くまで**動かずに待たされる**のか
- (b) 3本とも終わるまでに何 ms か。3本は**同時に終わる**のか、**1本ずつ順に終わる**のか
- (c) 同じことを「**ひたすら待つだけの仕事**」でやったら、答えは変わるか

In [ ]:
%%writefile adv01b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <chrono>
#include <atomic>
#include <mutex>
using namespace std::chrono;

std::atomic<long> sink{0};   // 複数スレッドから足すので atomic
std::mutex out_mtx;          // 表示が混ざらないように
steady_clock::time_point t0;
int ms_now() { return (int)duration_cast<milliseconds>(steady_clock::now() - t0).count(); }

long calib = 0;
long burn(long n) { long s = 0; for (long i = 0; i < n; i++) s += (i * 2654435761u) % 7; return s; }
void calibrate() {
    long n = 100000;
    for (;;) {
        auto a = steady_clock::now(); sink.fetch_add(burn(n), std::memory_order_relaxed);
        auto us = duration_cast<microseconds>(steady_clock::now() - a).count();
        if (us > 30000) { calib = n * 1000 / us; break; }
        n *= 2;
    }
}

// 3本のスレッドを立てて、それぞれが終わった時刻を出す
void run(const char* kind, bool cpu) {
    std::cout << "\n【" << kind << " を 3本】\n";
    t0 = steady_clock::now();
    std::vector<std::thread> ts;
    for (int k = 0; k < 3; k++)
        ts.emplace_back([=] {
            if (cpu) sink.fetch_add(burn(calib * 300), std::memory_order_relaxed);  // 300ms 分の計算
            else     std::this_thread::sleep_for(milliseconds(300));        // 300ms 待つだけ
            std::lock_guard<std::mutex> g(out_mtx);   // 表示だけ鍵の中（計算は外）
            std::cout << "  スレッド" << k << " が終わった : " << ms_now() << " ms\n";
        });
    for (auto& t : ts) t.join();
    std::cout << "  全部終わるまで : " << ms_now() << " ms\n";
}

int main() {
    calibrate();
    std::cout << "hardware_concurrency() = " << std::thread::hardware_concurrency() << "\n";
    run("計算しかしない仕事（1本あたり300ms分）", true);
    run("ひたすら待つだけの仕事（300ms）",        false);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread adv01b.cpp -o adv01b && ./adv01b

### 問1-3 の解答 ―― 3本とも少しずつ進み、ほぼ同時に終わる

```
【計算しかしない仕事 を 3本】
  スレッド0 が終わった : 411 ms
  スレッド1 が終わった : 450 ms
  スレッド2 が終わった : 468 ms
  全部終わるまで : 468 ms

【ひたすら待つだけの仕事 を 3本】
  スレッド0 が終わった : 300 ms
  スレッド1 が終わった : 300 ms
  スレッド2 が終わった : 300 ms
  全部終わるまで : 300 ms
```

**(a) 待たされません。** OS は短い時間で3本を切り替えながら実行します（タイムスライス）。
「3本目だけ止まっている」のではなく、**3本とも少しずつ進みます。**

**(b) 約 450ms** ―― `300ms × 3本 ÷ 2コア = 450ms` です。
そして **1本ずつ順にではなく、ほぼ同時に終わります**（切り替えながら均等に進むため）。

> 全部が終わる時間は変わらないのに、**どれ1つとして早くは終わりません。**
> 「1本目だけ先に終わらせて結果を使いたい」なら、この形は不利です。

**(c) 変わります。** 待つ仕事なら 3本とも 300ms で終わります。
待つのに計算回路は要らないので、コア数は関係ありません（演習1-3）。

> **計算する仕事は「同時に計算できる数」で頭打ち。待つ仕事は何本でも重なる。**